In [1]:
import sys
sys.path.append("../src")

In [ ]:
import pandas as pd
from pathlib import Path
from modelado import (asignar_estaciones, calcular_grados, filtrar_entrenamiento,
                      calcular_temperatura_media, armar_futuro,
                      pronosticar_cuantiles, maximo_estacional,
                      preparar_serie_hw, pronosticar_hw, pronostico_anterior_mensual,
                      calcular_maximos_estacionales, pronostico_ingenuo, fechas_estacion, pronostico_retrospectivo)
from regulacion import calcular_piso, asignar_piso, calcular_hrp, calcular_cargo, cargo_estacion, cargo_escenarios, sobrecosto_evitable

from evaluacion import errores_maximo, cargos_retrospectivos, resumen_metricas


In [3]:
ruta_retrospectivo = Path("../data/processed/RETROSPECTIVO_CTRELEUW.csv")
ruta_retrospectivo_mensual = Path("../data/processed/RETROSPECTIVO_MENSUAL_CTRELEUW.csv")
retrospectivo_mensual = pd.read_csv(ruta_retrospectivo_mensual, parse_dates=["MES"])
retrospectivo = pd.read_csv(ruta_retrospectivo)

In [4]:
retrospectivo

,ESTACION,ANIO_ESTACION,MAXIMO_MW,PRONOSTICO_ANT,Q10_MW,Q50_MW,Q90_MW,HW_MW
0,INVIERNO,2021,42.352,NaN,45.414681,46.027273,47.260814,40.114391
1,INVIERNO,2022,44.156,42.352,43.734627,45.302328,46.840297,42.563225
2,INVIERNO,2023,42.448,44.156,43.181230,45.017425,46.953116,42.098513
3,INVIERNO,2024,41.960,42.448,42.678967,44.313533,46.699444,42.394445
4,INVIERNO,2025,44.436,41.960,42.253762,43.863804,46.760655,42.071078
5,VERANO,2021,44.380,44.396,43.781919,44.340467,46.007473,41.569720
6,VERANO,2022,45.528,44.380,43.314707,44.314921,45.871326,40.944850
7,VERANO,2023,48.340,45.528,42.831921,44.551764,45.881774,42.461331
8,VERANO,2024,47.856,48.340,42.829853,45.065222,46.901773,42.200645
9,VERANO,2025,44.244,47.856,42.697954,45.233164,47.076460,43.833393


In [5]:
sin_na = retrospectivo["PRONOSTICO_ANT"].notna()
comunes = retrospectivo[sin_na].reset_index(drop=True).copy()
len(comunes)

9

In [6]:
comunes["ERROR_RC"] = comunes["MAXIMO_MW"] - comunes["Q50_MW"]
comunes["ERROR_HW"] = comunes["MAXIMO_MW"] - comunes["HW_MW"]
comunes["ERROR_ANT"] = comunes["MAXIMO_MW"] - comunes["PRONOSTICO_ANT"]

In [7]:
mae_rc = comunes["ERROR_RC"].abs().mean()
mae_hw = comunes["ERROR_HW"].abs().mean()
mae_ant = comunes["ERROR_ANT"].abs().mean()

In [8]:
print("MAE regresión cuantílica: ", round(mae_rc, 3))
print("MAE Holt-Winters: ", round(mae_hw, 3))
print("MAE repetición del pico anterior: ", round(mae_ant, 3))

MAE regresión cuantílica:  1.718
MAE Holt-Winters:  2.676
MAE repetición del pico anterior:  1.616


In [9]:
real = retrospectivo["MAXIMO_MW"]

In [10]:
por_debajo = (real < retrospectivo["Q10_MW"]).mean()
por_encima = (real > retrospectivo["Q90_MW"]).mean()

In [11]:
cobertura = ((real >= retrospectivo["Q10_MW"]) & (real <= retrospectivo["Q90_MW"])).mean()

In [12]:
print("Cobertura: ", cobertura)
print("Por debajo del 0.1: ", por_debajo)
print("Por encima del 0.9: ", por_encima)

Cobertura:  0.5
Por debajo del 0.1:  0.3
Por encima del 0.9:  0.2


In [13]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])

In [14]:
reales = serie[["MES", "MAX_REAL_MW"]]
retrospectivo_mensual = retrospectivo_mensual.merge(reales, on="MES", how="left")

In [15]:
retrospectivo_mensual = pronostico_anterior_mensual(serie, retrospectivo_mensual)

In [16]:
serie = asignar_estaciones(serie)
maximos = calcular_maximos_estacionales(serie)
maximos = calcular_piso(maximos)

In [17]:
retrospectivo_mensual = asignar_piso(retrospectivo_mensual, maximos)
retrospectivo_mensual.loc[retrospectivo_mensual["ANIO_ESTACION"].isin([2022, 2025]), ["ESTACION", "ANIO_ESTACION", "MES", "PISO_MW"]]

,ESTACION,ANIO_ESTACION,MES,PISO_MW
6,INVIERNO,2022,2022-05-01,38.117
7,INVIERNO,2022,2022-06-01,38.117
8,INVIERNO,2022,2022-07-01,38.117
9,INVIERNO,2022,2022-08-01,38.117
10,INVIERNO,2022,2022-09-01,38.117
11,INVIERNO,2022,2022-10-01,38.117
24,INVIERNO,2025,2025-05-01,37.764
25,INVIERNO,2025,2025-06-01,37.764
26,INVIERNO,2025,2025-07-01,37.764
27,INVIERNO,2025,2025-08-01,37.764


In [18]:
retrospectivo_mensual = calcular_hrp(retrospectivo_mensual)
retrospectivo_mensual.loc[retrospectivo_mensual["MES"] == "2025-11-01", ["MES", "HRP"]]

,MES,HRP
54,2025-11-01,360


In [19]:
escenarios = {"CARGO_REFERENCIA_USD": "MAX_REAL_MW", "CARGO_RC_USD": "PRED_RC_MW", "CARGO_HW_USD": "PRED_HW_MW", "CARGO_ANT_USD": "PRED_ANT_MW"}
retrospectivo_mensual = cargo_escenarios(retrospectivo_mensual, escenarios)
retrospectivo_mensual.loc[retrospectivo_mensual["MES"] == "2025-11-01", ["MAX_REAL_MW", "PRED_RC_MW", "PISO_MW", "HRP", "CARGO_REFERENCIA_USD", "CARGO_RC_USD"]]

,MAX_REAL_MW,PRED_RC_MW,PISO_MW,HRP,CARGO_REFERENCIA_USD,CARGO_RC_USD
54,38.628,40.666912,35.892,360,200247.55,210817.27


In [20]:
columnas_cargo = list(escenarios.keys())
cargos = retrospectivo_mensual.groupby(["ESTACION", "ANIO_ESTACION"])[columnas_cargo].sum(min_count=6).reset_index()
cargos

,ESTACION,ANIO_ESTACION,CARGO_REFERENCIA_USD,CARGO_RC_USD,CARGO_HW_USD,CARGO_ANT_USD
0,INVIERNO,2021,1517415.45,1654283.68,1462924.29,1531932.58
1,INVIERNO,2022,1560936.75,1614272.39,1548726.80,1523347.81
2,INVIERNO,2023,1564339.05,1572565.34,1544591.02,1580492.62
3,INVIERNO,2024,1510929.10,1555607.04,1547288.44,1552770.46
4,INVIERNO,2025,1569367.27,1544444.09,1537570.77,1505717.38
5,VERANO,2021,1544361.97,1557271.16,1504655.09,1558451.98
6,VERANO,2022,1574259.85,1526439.72,1490948.91,1533052.28
7,VERANO,2023,1615690.14,1563323.48,1547917.08,1595407.91
8,VERANO,2024,1649283.73,1617587.89,1617587.89,1663366.12
9,VERANO,2025,1580441.24,1531411.72,1520319.68,1530650.96


In [21]:
retrospectivo_mensual.loc[(retrospectivo_mensual["ESTACION"] == "VERANO") & (retrospectivo_mensual["ANIO_ESTACION"] == 2024), ["MES", "MAX_REAL_MW", "PRED_RC_MW", "PRED_HW_MW", "PRED_ANT_MW", "PISO_MW"]]

,MES,MAX_REAL_MW,PRED_RC_MW,PRED_HW_MW,PRED_ANT_MW,PISO_MW
48,2024-11-01,37.926,40.775220,37.960797,38.508,43.506
49,2024-12-01,40.441,41.870688,39.087672,37.004,43.506
50,2025-01-01,47.856,42.974850,41.923058,48.340,43.506
51,2025-02-01,40.894,42.323789,42.200645,45.172,43.506
52,2025-03-01,40.842,40.836117,38.243574,35.828,43.506
53,2025-04-01,37.499,38.415550,37.575607,38.104,43.506


In [22]:
for modelo in ["RC", "HW", "ANT"]:
    cargos[f"ERROR_{modelo}_USD"] = cargos["CARGO_REFERENCIA_USD"] - cargos[f"CARGO_{modelo}_USD"]
resumen_cargo = pd.DataFrame({
    "MAE_USD": cargos[["ERROR_RC_USD", "ERROR_HW_USD", "ERROR_ANT_USD"]].abs().mean(),
    "SESGO_USD": cargos[["ERROR_RC_USD", "ERROR_HW_USD", "ERROR_ANT_USD"]].mean()
}).round(2)
resumen_cargo


,MAE_USD,SESGO_USD
ERROR_RC_USD,46185.26,-5018.20
ERROR_HW_USD,43721.33,36449.46
ERROR_ANT_USD,31320.34,11183.45


In [23]:
regimen_nuevo = sobrecosto_evitable(serie, maximos, "2025-11-01", "2026-10-01")
sobrecosto = regimen_nuevo.groupby(["ESTACION", "ANIO_ESTACION"])[["CARGO_DECLARADO_USD", "CARGO_PISO_USD", "SOBRECOSTO_USD"]].sum().reset_index()
sobrecosto

,ESTACION,ANIO_ESTACION,CARGO_DECLARADO_USD,CARGO_PISO_USD,SOBRECOSTO_USD
0,INVIERNO,2026,1434626.73,1374333.69,60293.04
1,VERANO,2025,1580441.24,1580441.24,0.00


In [24]:
cargos[["ESTACION", "ANIO_ESTACION", "ERROR_RC_USD"]]

,ESTACION,ANIO_ESTACION,ERROR_RC_USD
0,INVIERNO,2021,-136868.23
1,INVIERNO,2022,-53335.64
2,INVIERNO,2023,-8226.29
3,INVIERNO,2024,-44677.94
4,INVIERNO,2025,24923.18
5,VERANO,2021,-12909.19
6,VERANO,2022,47820.13
7,VERANO,2023,52366.66
8,VERANO,2024,31695.84
9,VERANO,2025,49029.52
